# ReAct Agent with Groq (Free-tier API)

This notebook uses Groq instead of OpenAI. It keeps the **classic text-based ReAct loop** (`Thought → Action → Action Input → Observation`) so you can inspect how the agent chooses and runs tools.

**Before running:** add `GROQ_API_KEY`, `TAVILY_API_KEY`, and `WEATHERSTACK_API_KEY` to a `.env` file in the same directory. Groq API access is subject to model availability and free-tier rate limits.

## 1. Install dependencies

Run this cell once in your notebook environment. If you already have compatible versions installed, you can skip it and restart the kernel if imports fail.

In [ ]:
%pip install -U langchain-classic langchain-groq langchain-community langchain-core python-dotenv certifi requests tavily-python

## 2. Imports

In [ ]:
import os
import requests
import certifi

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_classic.agents import create_react_agent, AgentExecutor

# Keep HTTPS certificate verification configured for Python requests.
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

## 3. Load and check API keys

Never paste API keys directly into notebook code or commit `.env` to Git.

In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing. Add it to your .env file.")

if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY is missing. Add it to your .env file.")

if not WEATHERSTACK_API_KEY:
    raise ValueError("WEATHERSTACK_API_KEY is missing. Add it to your .env file.")

print("API keys found (values are hidden).")

## 4. Create the tools

The agent can use Tavily web search and the Weatherstack current-weather API. The weather tool checks HTTP/API errors instead of failing with a confusing `KeyError`.

In [ ]:
search_tool = TavilySearchResults(max_results=3)

@tool
def get_weather_data(city: str) -> str:
    """Get current temperature, weather description, and humidity for a city."""
    try:
        response = requests.get(
            "https://api.weatherstack.com/current",
            params={"access_key": WEATHERSTACK_API_KEY, "query": city},
            timeout=15,
        )
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as exc:
        return f"Weather request failed for {city}: {exc}"
    except ValueError:
        return "Weather service returned an invalid response."

    if "current" not in data:
        error_info = data.get("error", {}).get("info", "No current-weather data returned.")
        return f"Could not fetch weather for {city}: {error_info}"

    current = data["current"]
    descriptions = current.get("weather_descriptions") or ["Description unavailable"]
    location = data.get("location", {}).get("name", city)
    return (
        f"City: {location}\n"
        f"Temperature: {current.get('temperature', 'N/A')}°C\n"
        f"Weather: {descriptions[0]}\n"
        f"Humidity: {current.get('humidity', 'N/A')}%"
    )

## 5. Test the tools independently

Check each tool works before debugging the agent. These calls use your Tavily and Weatherstack API quotas.

In [ ]:
# Web-search smoke test
search_result = search_tool.invoke({"query": "capital of India"})
print(search_result)

# Weather smoke test
weather_result = get_weather_data.invoke({"city": "New Delhi"})
print(weather_result)

## 6. Configure the Groq model

`llama-3.1-8b-instant` is a fast, lightweight model and supports tool use on Groq. If Groq says the model ID is unavailable for your account, check the currently enabled IDs in the [Groq model list](https://console.groq.com/docs/models) and replace `GROQ_MODEL` below.

This notebook uses the **classic text-form ReAct parser**, not Groq's native function-calling agent. ReAct depends on the model following the requested output format, so the verbose trace in the next section is important.

In [ ]:
GROQ_MODEL = "llama-3.1-8b-instant"

llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0,
    api_key=GROQ_API_KEY,
    max_retries=2,
)

# Simple connectivity/model-ID test
test_response = llm.invoke("Reply with exactly: Groq connection works.")
print(test_response.content)

## 7. Define an explicit ReAct prompt

This avoids relying on `hub.pull("hwchase17/react")`, which adds an external dependency and can be a source of prompt/version confusion. The required variables are `tools`, `tool_names`, `input`, and `agent_scratchpad`.

In [ ]:
react_template = """You are a helpful assistant that can use tools to answer questions accurately.

You have access to these tools:
{tools}

Use exactly this format:

Question: the question to answer
Thought: briefly decide what to do next
Action: one of [{tool_names}]
Action Input: the input for that action
Observation: the tool result
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now know the final answer
Final Answer: the answer to the original question

Rules:
- When a tool is needed, output one Action and one Action Input.
- Action must exactly match one of [{tool_names}].
- Do not invent tool results.
- After receiving an Observation, decide whether another tool is needed.
- When finished, output Final Answer.

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(react_template)
print("Prompt variables:", prompt.input_variables)

## 8. Create the ReAct agent and executor

`stop_sequence=False` avoids relying on provider-specific stop-sequence handling. The executor still parses the ReAct `Action` / `Final Answer` format. If parsing errors occur, inspect the model's raw output and consider switching to a native tool-calling agent instead.

In [ ]:
tools = [search_tool, get_weather_data]

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
    stop_sequence=False,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=(
        "Your previous response did not follow the required ReAct format. "
        "Use `Action: <tool name>` and `Action Input: <input>` for a tool call, "
        "or `Final Answer: <answer>` when finished."
    ),
    max_iterations=6,
    return_intermediate_steps=True,
)

## 9. Run a ReAct test

This test explicitly asks the agent to use **both tools**, so you can confirm the action/observation loop is working. The verbose output should show tool calls and observations before the final answer.

In [ ]:
result = agent_executor.invoke({
    "input": (
        "Use web search to verify the capital of India. Then call the "
        "get_weather_data tool for New Delhi and report the current weather. "
        "Clearly separate the search result from the weather result."
    )
})

print("\nFINAL ANSWER:\n")
print(result["output"])

print("\nTOOLS USED:")
for action, observation in result.get("intermediate_steps", []):
    print(f"- {action.tool}: {action.tool_input}")

## 10. How to tell whether ReAct worked

Look at the executor's verbose trace. A successful run should show:

1. The model emits an `Action` that exactly matches a tool name.
2. The selected tool runs and returns an `Observation`.
3. The model uses the observation and either calls another tool or returns `Final Answer`.

If you get `OutputParserException`, the model did not follow the expected text format. That is a **ReAct formatting issue**, not necessarily an API-key issue. If you get `model_not_found` or `401`, check the model ID or key. If you get a rate-limit error, wait for the limit to reset or check your Groq console.

**Important:** Classic ReAct is a text-parsing approach. For a more robust Groq agent, native tool calling with LangChain's `create_agent` is generally preferable, but it is a different implementation from this classic ReAct demo.